# Capital Region First Ring Expressway (formerly as Seoul Outer Ring Expressway) — 3D label (spatial × temporal) regression demo

This notebook shows a spatio-temporal regression workflow where inputs X are (month, day-type, direction) and labels Y are a (segment × hour) grid.
- Features X: shape (B, D) where B is the number of (month, day-type, direction) groups; D is feature dim (e.g., 3).
- Labels Y: shape (B, S, T) where S is segment count, T is 24 hours.
- Model: `enn_torch` spatiotemporal model built with `Model` + `ModelConfig/PatchConfig`.
- Note: this is a shape/alignment demo; tweak hyperparameters for real training.


In [ ]:
import os
import re
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import torch
from openpyxl import load_workbook
from tensordict import PersistentTensorDict, TensorDict

from enn_torch import new_embedding, new_model, predict, train
from enn_torch.core.config import ModelConfig, PatchConfig
from enn_torch.core.system import get_device
from enn_torch.data.pipeline import Dataset

# 프로젝트 루트
PROJECT_DIR = Path(os.environ.get("PROJECT_DIR", str(Path.cwd()))).resolve()

# 1. 기준이 되는 기본 경로 설정
base_tmp = "/var/tmp"

# 2. 관련 환경 변수들을 딕셔너리 형태로 정의 (가능하면 torch import 전에)
torch_env_vars: dict[str, str] = {
    "TMPDIR": base_tmp,
    "TORCH_HOME": f"{base_tmp}/torch_home",
    "PYTORCH_KERNEL_CACHE_PATH": f"{base_tmp}/torch_kernel_cache",
    "TORCHINDUCTOR_CACHE_DIR": f"{base_tmp}/torchinductor_cache",
}

# ✅ tensordict import 전에 한 번 더 확실히(프로세스 레벨)
os.environ.setdefault("EXCLUDE_TD_FROM_PYTREE", "1")

# 3. 루프를 돌며 환경 변수 설정
for key, value in torch_env_vars.items():
    os.environ[key] = value
    try:
        os.makedirs(value, exist_ok=True)
    except Exception:
        pass


# ✅ tensordict import 이후에도 확실하게 PyTree 제외(경고 원천 제거)
try:
    from tensordict.nn.functional_modules import _exclude_td_from_pytree

    _exclude_td_from_pytree().set()
except Exception:
    pass


# -----------------------------------------------------------------------------
# Constants & Mappings
# -----------------------------------------------------------------------------
HOURS: list[str] = [f"{h:02d}시" for h in range(24)]
DAY2ID: dict[str, int] = {
    "월요일": 0,
    "화요일": 1,
    "수요일": 2,
    "목요일": 3,
    "금요일": 4,
    "토요일": 5,
    "일요일": 6,
}
DIR2ID: dict[str, int] = {"상행": 0, "하행": 1}


# -----------------------------------------------------------------------------
# Helper Functions
# -----------------------------------------------------------------------------
def setup_paths() -> tuple[Path, Path, Path, Path]:
    """경로 설정 (EC2/Jupyter)
    - PROJECT_DIR(환경변수) 또는 현재 작업 디렉토리를 프로젝트 루트로 사용
    - raw_data.xlsx는 PROJECT_DIR 아래에 있어야 함
    """
    drive_root = PROJECT_DIR
    excel_path_drive = drive_root / "raw_data.xlsx"
    if not excel_path_drive.exists():
        raise FileNotFoundError(
            f"raw_data.xlsx not found under {drive_root}. Upload or copy it there."
        )
    out_dir_drive = drive_root

    # 로컬 작업 디렉토리(캐시/임시파일)는 /var/tmp를 사용
    work_dir = Path("/var/tmp/enn_work")
    work_dir.mkdir(parents=True, exist_ok=True)

    # EC2에서는 굳이 복사하지 않고 원본을 그대로 읽는다(필요 시 아래를 copy2로 바꿔도 됨)
    excel_path_local = excel_path_drive

    return excel_path_drive, out_dir_drive, work_dir, excel_path_local


def _normalize_pasted_text(val: object) -> str:
    return (
        str(val)
        .replace("\r\n", "\n")
        .replace("\r", "\n")
        .replace("\\r\\n", "\n")
        .replace("\\n", "\n")
        .replace("\n", " ")
        .strip()
    )


def parse_sheet_meta(name: str) -> tuple[int, str]:
    """시트 이름에서 월, 요일 추출 (⚠️ '01월'의 '월' 때문에 오탐 방지)"""
    cleaned_name = _normalize_pasted_text(name)
    m = re.search(r"(\d+)월", cleaned_name)
    if not m:
        raise ValueError(f"Month not found in sheet: {cleaned_name}")
    month = int(m.group(1))

    name_clean = cleaned_name.replace(m.group(0), "")
    day_char = next((char for char in "월화수목금토일" if char in name_clean), None)
    if day_char is None:
        raise ValueError(f"Weekday not found in sheet: {cleaned_name}")
    return month, f"{day_char}요일"


def read_excel_polars(path: Path, sheet_name: str) -> pl.DataFrame:
    try:
        return pl.read_excel(source=str(path), sheet_name=sheet_name)
    except Exception as error:
        print(
            f"[Warn] Polars read failed ({sheet_name}), fallback to pandas: {error}",
            flush=True,
        )
        return pl.from_pandas(pd.read_excel(str(path), sheet_name=sheet_name))


def _env_truthy(name: str, default: bool = False) -> bool:
    raw = _normalize_pasted_text(os.environ.get(name, ""))
    if not raw:
        return bool(default)
    match raw.lower():
        case "1" | "true" | "yes" | "y" | "on":
            return True
        case "0" | "false" | "no" | "n" | "off":
            return False
        case _:
            return bool(default)


def get_model_config(device: torch.device, S: int, T: int) -> ModelConfig:
    patch = PatchConfig(
        is_cube=True,
        grid_size_3d=(S, T, 1),
        patch_size_3d=(1, 1, 1),
        use_padding=True,
    )

    match device.type:
        case "cuda":
            params = {
                "d_model": 1024,
                "heads": 16,
                "mlp_ratio": 4.0,
                "dropout": 0.02,
                "depth": 4,
                "latents": 128,
            }
        case _:
            cpu_tiny = _env_truthy("ENN_NOTEBOOK_CPU_TINY", default=False)
            if cpu_tiny:
                params = {
                    "d_model": 96,
                    "heads": 2,
                    "mlp_ratio": 2.0,
                    "dropout": 0.05,
                    "depth": 1,
                    "latents": 16,
                }
            else:
                params = {
                    "d_model": 256,
                    "heads": 2,
                    "mlp_ratio": 2.0,
                    "dropout": 0.05,
                    "depth": 2,
                    "latents": 32,
                }

    return ModelConfig(
        device=device,
        patch=patch,
        normalization_method="layernorm",
        d_model=params["d_model"],
        heads=params["heads"],
        mlp_ratio=params["mlp_ratio"],
        dropout=params["dropout"],
        drop_path=0.05,
        spatial_depth=params["depth"],
        temporal_depth=params["depth"],
        spatial_latents=params["latents"],
        temporal_latents=params["latents"],
        preset="spatiotemporal",
        compile_mode="max-autotune" if device.type == "cuda" else "disabled",
    )


def canonicalize_section(sec: object) -> str:
    clean = _normalize_pasted_text(sec)
    if not clean:
        return ""
    parts = [p.strip() for p in clean.split("-") if p.strip()]
    return "-".join(sorted(parts)) if len(parts) > 1 else clean


# -----------------------------------------------------------------------------
# Main Execution Logic
# -----------------------------------------------------------------------------
def main() -> None:
    EXCEL_DRIVE, OUT_DRIVE, WORK_DIR, EXCEL_LOCAL = setup_paths()

    wb = load_workbook(str(EXCEL_LOCAL), read_only=True, data_only=True)
    sheet_names = list(wb.sheetnames)
    wb.close()

    print(f"Total sheets: {len(sheet_names)}, Sample: {sheet_names[:5]}", flush=True)

    long_parts: list[pl.LazyFrame] = []

    for sh in sheet_names:
        try:
            df = read_excel_polars(EXCEL_LOCAL, sh)
            df = df.rename({c: str(c).strip() for c in df.columns})

            if not {"구간", "방향"}.issubset(df.columns):
                continue
            hour_cols_present = [c for c in df.columns if c in HOURS]
            if not hour_cols_present:
                continue

            month, day_kind = parse_sheet_meta(sh)

            if "노선" not in df.columns:
                df = df.with_columns(pl.lit("수도권제1순환선").alias("노선"))

            lf = (
                df.with_row_index(name="row_in_sheet")
                .select(["row_in_sheet", "노선", "구간", "방향"] + hour_cols_present)
                .lazy()
                .with_columns(
                    [pl.lit(month).alias("월"), pl.lit(day_kind).alias("일종")]
                )
                .unpivot(
                    index=["row_in_sheet", "노선", "구간", "방향", "월", "일종"],
                    on=hour_cols_present,
                    variable_name="시간",
                    value_name="지표",
                )
            )
            long_parts.append(lf)

        except Exception as error:
            print(f"[Warn] Skipping sheet {sh}: {error}", flush=True)

    if not long_parts:
        raise RuntimeError("No valid data found in sheets.")

    long_lf = (
        pl.concat(long_parts, how="vertical")
        .with_columns(
            [
                pl.col("시간").str.replace("시", "").cast(pl.Int64, strict=False),
                pl.col("지표").cast(pl.Float64, strict=False),
            ]
        )
        .drop_nulls("지표")
    )

    canonical_expr = (
        pl.col("구간")
        .str.split("-")
        .list.eval(pl.element().str.strip_chars())
        .list.sort()
        .list.join("-")
    )
    seg_key_expr = pl.concat_str(
        [pl.col("노선").str.strip_chars(), pl.lit("|"), canonical_expr]
    )

    long_lf = long_lf.with_columns(
        [
            pl.col("일종").replace(DAY2ID).cast(pl.Int64).alias("요일타입_id"),
            pl.col("방향").replace(DIR2ID).cast(pl.Int64).alias("방향_id"),
            canonical_expr.alias("canonical_section"),
            seg_key_expr.alias("seg_key"),
        ]
    )

    seg_table_lf = (
        long_lf.select(["seg_key", "노선", "canonical_section"])
        .unique()
        .sort("seg_key")
        .with_row_index("seg_idx")
        .with_columns(pl.col("seg_idx").cast(pl.Int64))
    )

    long_lf = long_lf.join(
        seg_table_lf.select(["seg_key", "seg_idx"]), on="seg_key", how="left"
    )

    seg_meta_df = seg_table_lf.collect()
    seg_key2id = dict(zip(seg_meta_df["seg_key"], seg_meta_df["seg_idx"]))

    long_df = long_lf.sort(
        ["월", "요일타입_id", "방향_id", "seg_idx", "시간"]
    ).collect()
    S, T = seg_meta_df.height, 24
    print(f"Dimensions: S={S}, T={T}", flush=True)

    group_cols = ["월", "요일타입_id", "방향_id"]
    groups_df = long_df.select(group_cols).unique().sort(group_cols)
    x_keys: list[tuple[int, int, int]] = [
        tuple(map(int, row)) for row in groups_df.to_numpy()
    ]
    B = len(x_keys)

    pivot_df = long_df.pivot(
        index=group_cols + ["seg_idx"],
        on="시간",
        values="지표",
        aggregate_function="first",
    )

    full_grid = groups_df.join(
        pl.DataFrame({"seg_idx": np.arange(S, dtype=np.int64)}), how="cross"
    )
    y_full = full_grid.join(pivot_df, on=group_cols + ["seg_idx"], how="left")

    # fill + ensure all hour cols exist
    missing_hours = set(map(str, range(T))) - set(y_full.columns)
    if missing_hours:
        y_full = y_full.with_columns(
            [pl.lit(0.0).alias(h).cast(pl.Float32) for h in missing_hours]
        )

    y_full = y_full.with_columns(
        [pl.col(str(h)).fill_null(0.0).cast(pl.Float32) for h in range(T)]
    )

    Y_np = (
        y_full.sort(group_cols + ["seg_idx"])
        .select([str(h) for h in range(T)])
        .to_numpy()
    )
    Y_np = Y_np.reshape((B, S, T)).astype(np.float32, copy=True)  # ✅ writable 보장

    row_map = long_df.group_by(group_cols + ["seg_idx"]).agg(
        pl.col("row_in_sheet").min()
    )
    row_ids_np = (
        full_grid.join(row_map, on=group_cols + ["seg_idx"], how="left")
        .sort(group_cols + ["seg_idx"])
        .select(pl.col("row_in_sheet").fill_null(-1))
        .to_numpy()
        .reshape((B, S))
        .astype(np.int64, copy=True)  # ✅ writable 보장
    )

    td_train = TensorDict(
        {
            "X": torch.tensor(x_keys, dtype=torch.float32),
            "Y": torch.from_numpy(Y_np),
            "row_ids": torch.from_numpy(row_ids_np),
        },
        batch_size=[B],
    )

    print(f"Train Batch: {td_train.shape}", flush=True)

    device = get_device()
    print(f"Device: {device}", flush=True)

    ds = Dataset.for_device(device)
    feats, _, _, label_shape = ds.preprocess(td_train)

    config = get_model_config(device, S, T)
    embedding = new_embedding(
        int(feats.shape[1]),
        embedding={
            "continuous_idx": [],
            "categorical": [
                {
                    "name": "month",
                    "idx": 0,
                    "num_embeddings": 12,
                    "embedding_dim": 4,
                    "offset": -1,
                    "clamp": True,
                },
                {
                    "name": "weekday",
                    "idx": 1,
                    "num_embeddings": 7,
                    "embedding_dim": 3,
                    "clamp": True,
                },
                {
                    "name": "direction",
                    "idx": 2,
                    "num_embeddings": 2,
                    "embedding_dim": 2,
                    "clamp": True,
                },
            ],
        },
    )
    model = new_model(
        in_dim=feats.shape[1], out_shape=label_shape, config=config, embedding=embedding
    ).to(device)

    match device.type:
        case "cuda":
            epochs, base_lr = 100, 3e-3
        case _:
            epochs, base_lr = 50, 3e-3
    epoch_override = _normalize_pasted_text(os.environ.get("ENN_NOTEBOOK_EPOCHS", ""))
    if epoch_override:
        epochs = int(epoch_override)

    model = train(
        model,
        td_train,
        epochs=epochs,
        checkpoint=True,
        base_lr=base_lr,
        weight_decay=1e-4,
        val_frac=0.1,
    )

    PRED_H5_LOCAL = str(WORK_DIR / "predictions.h5")
    predict(
        model,
        td_train.select("X"),
        output="file",
        path=PRED_H5_LOCAL,
        overwrite="replace",
    )

    pred_td_ro = PersistentTensorDict(filename=PRED_H5_LOCAL, mode="r")
    X_pred = pred_td_ro["X"].cpu().numpy()
    PRED_KEY2IDX = {tuple(map(int, row)): i for i, row in enumerate(X_pred)}

    OUTPUT_XLSX_LOCAL = str(WORK_DIR / "pred_results.xlsx")

    with pd.ExcelWriter(OUTPUT_XLSX_LOCAL, engine="xlsxwriter") as writer:
        for sh in sheet_names:
            df_raw = pd.read_excel(str(EXCEL_LOCAL), sheet_name=sh)
            month, day_kind = parse_sheet_meta(sh)

            if "노선" in df_raw.columns:
                route = df_raw["노선"].astype(str).str.strip()
            else:
                route = pd.Series(["수도권제1순환선"] * len(df_raw))
            seg_keys = route + "|" + df_raw["구간"].map(canonicalize_section)
            seg_idxs = seg_keys.map(seg_key2id)

            df_out = df_raw.copy()
            for dir_name, dir_id in DIR2ID.items():
                gkey = (month, DAY2ID[day_kind], dir_id)
                idx_in_pred = PRED_KEY2IDX.get(gkey)
                if idx_in_pred is None:
                    continue

                mask = (df_out["방향"] == dir_name) & seg_idxs.notna()
                if not mask.any():
                    continue

                valid_rows_seg_idx = seg_idxs[mask].astype(int).to_numpy().copy()
                pred_vals = (
                    pred_td_ro[int(idx_in_pred)]["Y"][valid_rows_seg_idx, :24]
                    .cpu()
                    .numpy()
                )

                df_out.loc[mask, HOURS] = pred_vals

            df_out.to_excel(writer, sheet_name=sh, index=False)

    print(f"Saved Local: {OUTPUT_XLSX_LOCAL}", flush=True)
    pred_td_ro.close()

    shutil.copyfile(OUTPUT_XLSX_LOCAL, OUT_DRIVE / "pred_results_like_raw.xlsx")
    shutil.copyfile(PRED_H5_LOCAL, OUT_DRIVE / "predictions.h5")

    Path(OUTPUT_XLSX_LOCAL).unlink(missing_ok=True)
    Path(PRED_H5_LOCAL).unlink(missing_ok=True)

    # cleanup
    print("Cleaning up directories...", flush=True)
    if WORK_DIR.exists():
        shutil.rmtree(WORK_DIR, ignore_errors=True)
    for key, value in torch_env_vars.items():
        if key == "TMPDIR":
            continue
        p = Path(value)
        if p.exists():
            shutil.rmtree(p, ignore_errors=True)
            print(f"Deleted: {p}", flush=True)

    print("\n[Done] All processes finished.", flush=True)


if __name__ == "__main__":
    main()
